# CANS Data Cleaning Pipeline — Walkthrough

This notebook documents every data cleaning and preparation step that produces `merged_table_w_demos` — the analysis-ready dataset used in all downstream modeling and visualization.

**Input data:**
| File | Description |
|---|---|
| `CANS.xlsx` | Long-format assessment records: one row per CANS item per assessment per client |
| `Demographics.xlsx` | Client demographic attributes |
| `Incidents.xlsx` | Incident reports linked to clients by `OptionsNumber` |
| `UPDATED Incident Matching - Exact_Column_Name_Table.csv` | Reference table mapping raw incident type strings to canonical categories |
| `category_mapping.csv` | Alternative researcher-defined grouping of CANS items |

**Output:** `merged_table_w_demos` — one row per assessment per client, containing:
- Imputed CANS item scores (wide format)
- Official and alternative domain-level sum scores
- Dummy-encoded demographics
- Time-to-event and event-status columns for each incident category

**Data Cleaning Choices**

- Duplicates: There are 2836 duplicate rows in the CANS assessments distributed across 39 individuals these were dropped keeping the first record arbituarily. 
- The granularity of the CANS data is of item, a question on CANS assessment. There are 439 items where the ChoiceValue is null distributed across 82 unique individuals. I impute them with the mode of their CANSCategory. 
- CANS also are defined as having possible values of 0 to 3. However, there are 36 examples where the CANS takes on 1.5 and items where the value is instead a 4. I an attempt to infer the the data enterers meaning I round the 1.5s to 2s and round down the 4 to a 3. 
- There are a 143 items distributed across 20 people who have different values for the same CANSCategoryName and ItemTitle. For these I choose mode aswell.  
- There are 3528 items across 744 unique individuals who have the same items mapped to different CANSCateogrys with different ChoiveValues. In these cases I take the value assigned to the Category most frequently matched with this item. 
- The incidents table contained 174 exact duplicates which I dropped keeping the first. 
- **Left Censoring:** In this case there are examples of incidents occuring before the first assessment for example 144 unique individuals had atleast 1 incident before their first recorded CANS assessment but only 106 had their first incident after their first assessment. This can be described as a kind of left censoring where events are excluded because they occur before we know the factors of interest and therefore cannot coeherently think about the effect of the observed factors on the incidents occurence. 
- Choose to focus on the top 43 CANs scores that make up the primary assessment and not the extension questions due to much higher null rates.
- There are a subset of 10 individuals for whom 40 out of the 43 top CANS are null. These 10 individuals have been excluded since no inference can be made about the relationship between their incident risk and CANS. Any bias introduced by their admission will likely be small since these 10 ideas make up less than 1 percent of the sample. 


## 1. Imports

In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict
import seaborn as sns
import matplotlib.pyplot as plt

## 2. Load Raw Data

We load three Excel files. The CANS file is in **long format** — each row represents a single item scored during a single assessment for a single client. The incidents file contains one row per reported incident.

In [2]:
BASE      = '/Users/daniellancet/Desktop/Spring_2026/CDSS_170/CANS_INCIDENTS_PROJECT/Aspiranet_Data'
NOVEL     = '/Users/daniellancet/Desktop/Spring_2026/CDSS_170/CANS_INCIDENTS_PROJECT/Novel_Data'

cans_data  = pd.read_excel(f'{BASE}/CANS.xlsx')
demos_data = pd.read_excel(f'{BASE}/Demographics.xlsx')
incidents  = pd.read_excel(f'{BASE}/Incidents.xlsx')

print(f'CANS rows:      {len(cans_data):,}')
print(f'Demographics:   {len(demos_data):,}')
print(f'Incidents rows: {len(incidents):,}')

CANS rows:      260,988
Demographics:   3,878
Incidents rows: 10,395


## 3. Filter Incidents to Clients with CANS Records

Not every client with an incident has a CANS assessment, and vice versa. We restrict the incidents table to clients who appear in the CANS data so that all incident records can eventually be joined to CANS scores. We also drop any exact duplicate incident rows at this stage.

In [3]:
incidents = incidents.drop_duplicates(keep='first')
incidents = incidents[incidents['OptionsNumber'].isin(cans_data['OptionsNumber'].unique())]

print(f'Incidents after filter: {len(incidents):,}')

Incidents after filter: 1,956


## 4. Deduplicate CANS Records

The raw CANS dataset contains **2,836 exact duplicate rows** distributed across 39 individuals — likely caused by data entry errors or export artifacts. We drop all duplicates and keep the first occurrence. Because the rows are identical, the choice of which to keep is arbitrary.

In [4]:
n_dups             = cans_data.duplicated().sum()
n_clients_affected = cans_data[cans_data.duplicated()]['OptionsNumber'].nunique()
print(f'Duplicate rows: {n_dups:,} across {n_clients_affected} clients')

cans_data = cans_data.drop_duplicates(keep='first')
print(f'CANS rows after deduplication: {len(cans_data):,}')

Duplicate rows: 2,836 across 39 clients
CANS rows after deduplication: 258,152


## 5. Impute Missing `ChoiceValue`

`ChoiceValue` is the scored rating (0–3) for each CANS item. Some entries are missing. We fill each missing value with the **mode** (most common score) of other records in the same `CANSCategoryName` group. Using the mode preserves the discrete 0–3 scale rather than introducing fractional values that the mean would produce.

In [5]:
cans_data['ChoiceValue'] = cans_data.groupby('CANSCategoryName')['ChoiceValue'].transform(
    lambda x: x.fillna(x.mode()[0])
)

## 6. Normalize Item-to-Category Mapping

A small number of CANS items are inconsistently assigned to more than one `CANSCategoryName` across records — a data entry inconsistency. We resolve this by computing the **most frequently observed category for each item** and using that as the canonical mapping, overwriting any inconsistent labels.

In [6]:
# For each ItemTitle, find the CANSCategoryName that appears most often
item_to_category = (
    cans_data.groupby(['ItemTitle', 'CANSCategoryName'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .drop_duplicates(subset='ItemTitle', keep='first')
    .set_index('ItemTitle')['CANSCategoryName']
)

item_title_mapping = item_to_category  # alias used in imputation steps below
cans_data['CANSCategoryName'] = cans_data['ItemTitle'].map(item_to_category)

## 7. Coarsen Ethnicity Labels

The raw `Ethnicity` field contains ~20 fine-grained labels (e.g., `'white - central american'`, `'japanese'`, `'mexican'`). Many of these groups are too small for statistical analysis. We consolidate them into **6 broader categories**:

| Group | Includes |
|---|---|
| White | white, white - european, white - armenian, etc. |
| Latino | hispanic/latino, mexican, spanish, central american |
| Black | black |
| Asian | filipino, japanese, chinese, asian indian, other asian, other pacific islander |
| Native | american indian |
| Other | no preference, [not entered], and all unmapped values |

We define a helper function so the same logic can be reused if needed.

In [7]:
def consolidate_ethnicity(ethnicity_series):
    ethnicity_map = {
        'white': 'White',
        'white - central american': 'White',
        'white - middle eastern': 'White',
        'white - armenian': 'White',
        'white - european': 'White',
        'spanish': 'Latino',
        'hispanic/latino': 'Latino',
        'mexican': 'Latino',
        'central american': 'Latino',
        'black': 'Black',
        'filipino': 'Asian',
        'japanese': 'Asian',
        'chinese': 'Asian',
        'asian indian': 'Asian',
        'other asian': 'Asian',
        'no preference': 'Other',
        'american indian': 'Native',
        'other pacific islander': 'Asian',
        '[not entered]': 'Other',
    }
    return ethnicity_series.astype(str).str.strip().str.lower().map(ethnicity_map).fillna('Other')

## 8. Aggregate to Item-Assessment Level

The raw data may still have multiple rows per (client, date, item) group. We aggregate using the **median** `ChoiceValue` within each group, producing a clean long-format table with exactly **one row per item per assessment per client**.

We apply the ethnicity consolidation at this stage since the `Ethnicity` column is available at this granularity.

In [8]:
df = (
    cans_data
    .groupby([
        'OptionsNumber', 'DateCompleted', 'CANSCategoryName', 'ItemTitle',
        'QuestionKey', 'AgeWhenAssessed', 'Gender', 'Ethnicity',
        'PrimaryLanguage', 'Religion', 'County'
    ])['ChoiceValue']
    .median()
    .reset_index()
)

df['Coarsened_Ethnicity'] = consolidate_ethnicity(df['Ethnicity'].copy())

print(f'Long-format rows: {len(df):,}')

Long-format rows: 240,864


## 9. Clamp Out-of-Range `ChoiceValue`s

The CANS uses a **0–3 integer scale**. After the median aggregation, some values are fractional (1.5, 2.5) or outside the valid range (0.5, 4.0). We snap these to the nearest valid integer:

| Raw value | Corrected |
|---|---|
| 0.5 | 1 |
| 1.5 | 2 |
| 2.5 | 2 |
| 4.0 | 3 |

In [9]:
corrections = {0.5: 1, 1.5: 2, 2.5: 2, 4.0: 3}
for bad_val, good_val in corrections.items():
    df.loc[np.isclose(df['ChoiceValue'], bad_val), 'ChoiceValue'] = good_val

## 10. Categorize and Standardize Incident Types

The incident records use heterogeneous type labels collected over time. We standardize them in three steps:

1. **Replace vague umbrella types** (`'Unusual Incident'`, `'Other (including HIPAA breaches)'`) with their more specific `IncidentSubType` values.
2. **Map all types to canonical categories** using a reference lookup table (`UPDATED Incident Matching - Exact_Column_Name_Table.csv`). Each column of that table is a canonical category, and its cells list all raw type strings that belong to it.
3. Any type not covered by the table is assigned to **'Other Incidents'**.

In [10]:
# Step 1: Replace vague top-level types with their subtypes
vague_types = ['Unusual Incident', 'Other (including HIPAA breaches)']
mask = incidents['IncidentType'].isin(vague_types)
incidents.loc[mask, 'IncidentType'] = incidents.loc[mask, 'IncidentSubType']

# Step 2: Build lookup dict {raw_type: canonical_category} from the reference table
incident_matching_table = pd.read_csv(
    f'{NOVEL}/UPDATED Incident Matching - Exact_Column_Name_Table.csv'
)
incident_matching_dict = {}
for col in incident_matching_table.columns:
    for incident in incident_matching_table[col].dropna():
        incident_matching_dict[incident] = col

# Step 3: Assign uncovered types to 'Other Incidents'
other_incidents = [
    '51. Other (including HIPAA breaches)', 'Other', 'Epidemic Outbreak',
    'Noise Complaints', 'Apartment Break in', 'Car Accident', 'Restrained by FP',
    'Unwanted Guests', 'zz. (obsolete) Crisis Center Visit (use reason for',
    'Fire/Explosion', 'COVID', 'Arrest', '06. Infection Control/Epidemic Outbreak Issue',
]
for v in other_incidents:
    incident_matching_dict[v] = 'Other Incidents'

# Apply mapping and normalize two special category name strings
incidents['IncidentCategory'] = (
    incidents['IncidentType']
    .apply(lambda x: incident_matching_dict.get(x, x))
    .replace('Police involvement', 'Police_Involvement')
    .replace('Inappropriate Sexual Behavior', 'Behavioral_Incidents')
)

## 11. Build First-Incident-Date Table

For each incident category we find the **earliest incident date** per client. This gives us one column per category (`first_incident_date_[category]`). Clients with no incident in a given category receive `NaT`. We then merge this wide table onto the main CANS dataframe on `OptionsNumber`.

In [11]:
incident_cols = {}
for category in incident_matching_table.columns:
    first_incident = (
        incidents[incidents['IncidentCategory'] == category]
        .sort_values('IncidentDate')
        .groupby('OptionsNumber')['IncidentDate']
        .first()
    )
    incident_cols[f'first_incident_date_{category}'] = first_incident

incidents_table = pd.DataFrame(incident_cols).reset_index()
df = df.merge(incidents_table, on='OptionsNumber', how='left')

print(f'First-incident columns added: {len(incident_cols)}')

First-incident columns added: 8


## 12. Compute Assessment Timing Features

Survival analysis requires knowing when each assessment occurred relative to the client's full history. We derive five timing columns:

| Column | Description |
|---|---|
| `origin_assessment` | Client's **first** CANS assessment date — used as the survival time origin |
| `last_assessment` | Client's **most recent** assessment date |
| `UniqueDatesPerOption` | Number of distinct assessment dates for the client |
| `prior_assessment_date` | Date of the immediately preceding assessment (NaT for first assessment) |
| `assessment_number` | Sequential index: 1 = first assessment, 2 = second, etc. |

We compute `prior_assessment_date` and `assessment_number` from the deduplicated (client, date) pairs to avoid inflating counts from the long-format rows.

In [12]:
df['origin_assessment']    = df.groupby('OptionsNumber')['DateCompleted'].transform('min')
df['last_assessment']      = df.groupby('OptionsNumber')['DateCompleted'].transform('max')
df['UniqueDatesPerOption'] = df.groupby('OptionsNumber')['DateCompleted'].transform('nunique')

df = df.sort_values(['OptionsNumber', 'DateCompleted'])

# Operate on unique (client, date) pairs to avoid duplicating shifts across long-format rows
prior_dates = (
    df[['OptionsNumber', 'DateCompleted']]
    .drop_duplicates()
    .sort_values(['OptionsNumber', 'DateCompleted'])
)
prior_dates['prior_assessment_date'] = prior_dates.groupby('OptionsNumber')['DateCompleted'].shift(1)
prior_dates['assessment_number']     = prior_dates.groupby('OptionsNumber').cumcount() + 1

df = (
    df
    .drop(columns=['prior_assessment_date', 'Religion'], errors='ignore')
    .merge(prior_dates, on=['OptionsNumber', 'DateCompleted'], how='left')
)

## 13. Left-Censor Pre-Baseline Incidents

Incidents that occurred **before** a client's first CANS assessment cannot be linked to any CANS scores — they pre-date our observation window entirely. Retaining them would introduce **left-censoring bias** by making it appear that an incident followed a set of scores that were actually recorded after the fact. We null out any `first_incident_date_*` value that is earlier than `origin_assessment`.

In [13]:
incident_date_cols = [c for c in df.columns if c.startswith('first_incident_date_')]
for col in incident_date_cols:
    df.loc[df[col] < df['origin_assessment'], col] = pd.NaT

## 14. Filter to Key CANS Domains

The full CANS instrument covers many sub-scales. We restrict to the **5 domains** most relevant to this analysis:

- Life Functioning Domain
- Caregiver Strengths & Needs
- Strengths
- Risk Behaviors
- Emotional/Behavioral/Mental Health Needs

A small number of clients may be dropped here if all of their item records fall outside these domains.

In [14]:
cat_list = [
    'Life Functioning Domain',
    'Caregiver Strengths & Needs',
    'Strengths',
    'Risk Behaviors',
    'Emotional/Behavioral/Mental Health Needs',
]
df_cat = df[df['CANSCategoryName'].isin(cat_list)]

lost_clients = set(df['OptionsNumber']) - set(df_cat['OptionsNumber'])
print(f'Clients lost by domain filter: {len(lost_clients)}')

Clients lost by domain filter: 10


## 15. Pivot to Wide Format and Select Items

We convert from long format (one row per item per assessment) to **wide format** (one row per assessment, one column per CANS item). We then select the **45 items with the fewest missing values** to ensure the dataset is dense enough for reliable imputation and modeling.

In [16]:
pivot = df_cat.pivot_table(
    index=['OptionsNumber', 'DateCompleted'],
    columns='ItemTitle',
    values='ChoiceValue',
).reset_index()

# Select the 45 CANS items with the fewest NaNs (skip the two index columns)
n = 45
cans_cols = list(pivot.isna().sum().sort_values(ascending=True).head(n + 2).index)[2:]

index_cols = ['OptionsNumber', 'DateCompleted']
cans_pivot_table = pivot[index_cols + cans_cols]

print(f'Wide-format shape: {cans_pivot_table.shape}')

Wide-format shape: (3383, 47)


## 16. Three-Stage CANS Imputation

Even after selecting the densest items, some cells remain missing. We apply a **three-stage imputation strategy**, where each stage fills what the previous one could not:

| Stage | Method | Rationale |
|---|---|---|
| 1 | **Historical median** | For each client + item, use the expanding median of the client's own past assessments. Highest-quality because it uses the client's actual trajectory. |
| 2 | **Category peer median** | For each row, fill remaining gaps with the median of other items in the same CANS domain. Exploits within-domain correlation. |
| 3 | **Global row median** | Final fallback: fill any remaining NaN with the median across all available CANS scores in that assessment row. |

In [17]:
def impute_historical_median(df):
    """Fill NaNs using each client's own expanding (cumulative) median over past assessments."""
    df_sorted = df.sort_values(['OptionsNumber', 'DateCompleted'])
    cols_to_fill = df_sorted.columns.difference(['OptionsNumber', 'DateCompleted'])
    for col in cols_to_fill:
        past_median = df_sorted.groupby('OptionsNumber')[col].transform(
            lambda x: x.expanding().median().shift()
        )
        df_sorted[col] = df_sorted[col].fillna(past_median)
    return df_sorted


def category_impute(df, item_to_cat_dict):
    """Fill NaNs with the row-wise median of other items in the same CANS domain."""
    df_imputed = df.copy()
    cat_to_items = defaultdict(list)
    for item, cat in item_to_cat_dict.items():
        if item in df.columns:
            cat_to_items[cat].append(item)

    for cat, items in cat_to_items.items():
        if len(items) > 1:
            row_peer_median = df_imputed[items].median(axis=1)
            for item in items:
                df_imputed[item] = df_imputed[item].fillna(row_peer_median)
    return df_imputed


def row_median_impute(df, cans_cols):
    """Final fallback: fill remaining NaNs with the row-wise median across all CANS items."""
    df_imputed = df.copy()
    global_row_median = df_imputed[cans_cols].median(axis=1)
    for col in cans_cols:
        df_imputed[col] = df_imputed[col].fillna(global_row_median)
    return df_imputed


def impute_missing_values(df, item_title_mapping, cans_cols):
    """Apply three-stage imputation: historical median → category peer → global row median."""
    df = impute_historical_median(df)
    df = category_impute(df, item_title_mapping)
    df = row_median_impute(df, cans_cols)
    return df


cans_table = impute_missing_values(
    cans_pivot_table,
    item_title_mapping=item_title_mapping,
    cans_cols=cans_cols,
)
print(f'Missing values after imputation: {cans_table[cans_cols].isna().sum().sum()}')

Missing values after imputation: 0


## 17. Compute Official CANS Domain Scores

For each of the 5 CANS domains, we **sum all item scores** within that domain to produce a single domain-level score. Column names follow the pattern `[Domain]_score` (spaces and slashes replaced with underscores). Higher scores indicate greater need — or, for the Strengths domain, greater strength.

In [19]:
cat_to_items = defaultdict(list)
for item, cat in item_to_category.items():
    if item in cans_table.columns:
        cat_to_items[cat].append(item)

for cat, items in cat_to_items.items():
    col_name = cat.replace(' ', '_').replace('/', '_') + '_score'
    cans_table[col_name] = cans_table[items].sum(axis=1)

category_score_cols = [c for c in cans_table.columns if c.endswith('_score') and not c.startswith('alt_')]
print(f'Domain score columns: {category_score_cols}')

Domain score columns: ['Life_Functioning_Domain_score', 'Emotional_Behavioral_Mental_Health_Needs_score', 'Risk_Behaviors_score', 'Strengths_score', 'Caregiver_Strengths_&_Needs_score']


## 18. Compute Alternative Category Scores

We load a researcher-defined alternative grouping of CANS items (`category_mapping.csv`) that clusters items into thematic groups distinct from the official CANS domains. These `alt_*_score` columns enable exploratory analysis and robustness checks against different grouping assumptions.

In [20]:
cans_mapping     = pd.read_csv(f'{NOVEL}/category_mapping.csv')
cans_mapping_dict = dict(zip(cans_mapping['variable'], cans_mapping['group']))

alt_cat_to_items = defaultdict(list)
for item, cat in cans_mapping_dict.items():
    if item in cans_table.columns:
        alt_cat_to_items[cat].append(item)

for cat, items in alt_cat_to_items.items():
    col_name = 'alt_' + cat.replace(' ', '_').replace('/', '_') + '_score'
    cans_table[col_name] = cans_table[items].sum(axis=1)

alt_score_cols = [c for c in cans_table.columns if c.startswith('alt_') and c.endswith('_score')]
print(f'Alternative score columns: {alt_score_cols}')

Alternative score columns: ['alt_Caregiver_Support_Needs_score', 'alt_Internalizing___Self-Harm_score', 'alt_Developmental_&_Sexual_Concerns_score', 'alt_Externalizing_Behavior_score', 'alt_Family_&_Social_Functioning_score', 'alt_Community_&_Strengths_score', 'alt_Substance_Use_&_Delinquency_score', 'alt_School_Functioning_score']


## 19. Merge Assessment-Level Features

`df` is still in long format (one row per item per assessment). We collapse it to **assessment level** by taking the first value of each demographic and incident-date column per (client, date) group — these values are constant within a group. We then merge with the wide-format CANS score table.

In [22]:
columns = [
    'AgeWhenAssessed',
    'Gender',
    'Coarsened_Ethnicity',
    'PrimaryLanguage',
    'County',
    'first_incident_date_Behavioral_Incidents',
    'first_incident_date_Health/Medical_Incidents',
    'first_incident_date_Self_Harm',
    'first_incident_date_Placement_Changes',
    'first_incident_date_Police_Involvement',
    'first_incident_date_Abuse/CPS Report',
    'first_incident_date_Suicide Related Incidents',
    'first_incident_date_AWOL/ Child_Absense',
    'origin_assessment',
    'UniqueDatesPerOption',
    'last_assessment',
    'prior_assessment_date',
    'assessment_number',
]

assessment_lvl_tbl = df.groupby(['OptionsNumber', 'DateCompleted'])[columns].first().reset_index()
merged_table = assessment_lvl_tbl.merge(cans_table, on=['OptionsNumber', 'DateCompleted'])

print(f'Merged table shape: {merged_table.shape}')

Merged table shape: (3383, 78)


## 20. Dummy-Encode Categorical Demographics

Cox regression and other models require numeric inputs. We one-hot encode three categorical variables, dropping one reference category from each to avoid multicollinearity (the dummy trap):

| Variable | Reference (dropped) | Rationale |
|---|---|---|
| Gender | M | Largest group |
| Coarsened_Ethnicity | White | Largest group |
| PrimaryLanguage | English | Largest group |

In [24]:
def process_categorical(df, cols, reference_categories=None):
    """One-hot encode categorical columns, dropping a specified reference category from each."""
    for col in cols:
        ref = reference_categories.get(col) if reference_categories else None
        col_dummies = pd.get_dummies(df[col], dtype=int)

        if ref is not None:
            if ref not in col_dummies.columns:
                raise ValueError(
                    f"Reference category '{ref}' not found in column '{col}'. "
                    f"Available: {list(col_dummies.columns)}"
                )
            col_dummies = col_dummies.drop(columns=[ref])
        else:
            col_dummies = col_dummies.iloc[:, 1:]  # drop first as fallback

        df = pd.concat([df, col_dummies], axis=1).drop(columns=[col])
    return df


cat_cols = ['Gender', 'Coarsened_Ethnicity', 'PrimaryLanguage']
binary_cat_cols = process_categorical(
    merged_table[cat_cols].copy(),
    cols=cat_cols,
    reference_categories={'Coarsened_Ethnicity': 'White', 'Gender': 'M', 'PrimaryLanguage': 'English'},
)

merged_table_w_demos = pd.concat(
    [merged_table.drop(columns=cat_cols), binary_cat_cols],
    axis=1,
)

## 21. Create Time-to-Event and Status Columns

For each incident category we create two columns required by survival models (e.g., Cox proportional hazards):

- **`T_[category]`** — time in days from `origin_assessment` to either (a) the incident date or (b) the censoring time.
- **`status_[category]`** — 1 if the incident occurred, 0 if the observation was censored.

**Censoring rule for clients with no incident:** we use the *earlier* of:
- `last_assessment + 200 days` — a buffer that accounts for the lag between the last recorded assessment and when outcomes are typically documented.
- `2026-01-01` — the data collection cutoff, beyond which no incidents were recorded.

In [25]:
STUDY_CUTOFF = pd.Timestamp('2026-01-01')

for col in merged_table_w_demos.filter(like='first_incident_date_').columns:
    suffix = col.replace('first_incident_date_', '')

    days_to_event = (merged_table_w_demos[col] - merged_table_w_demos['origin_assessment']).dt.days
    status        = (~merged_table_w_demos[col].isna()).astype(int)

    # Censoring time: min(last_assessment + 200-day buffer, study cutoff)
    last_plus_buffer = (
        (merged_table_w_demos['last_assessment'] - merged_table_w_demos['origin_assessment']).dt.days + 200
    )
    cutoff_days  = (STUDY_CUTOFF - merged_table_w_demos['origin_assessment']).dt.days
    censor_time  = np.minimum(last_plus_buffer, cutoff_days)

    merged_table_w_demos[f'T_{suffix}']      = days_to_event.fillna(censor_time)
    merged_table_w_demos[f'status_{suffix}'] = status

print('Time-to-event columns created.')

Time-to-event columns created.


## 22. Final Dataset

`merged_table_w_demos` is the fully cleaned, analysis-ready dataset. 

Each row represents **one assessment for one client** and contains:
- Imputed CANS item scores (wide format, 45 items)
- Official domain sum scores (`*_score`)
- Alternative thematic sum scores (`alt_*_score`)
- Dummy-encoded demographics (Gender, Ethnicity, Language)
- Time-to-event (`T_*`) and event-status (`status_*`) columns for each incident category

In [36]:
print(f'Final dataset shape: {merged_table_w_demos.shape}')
print(f'Unique clients:      {merged_table_w_demos["OptionsNumber"].nunique():,}')


Final dataset shape: (3383, 98)
Unique clients:      1,063
